# Chapter 43 — Production: Serving, Drift, and MLOps

*From Absolute Zero* — companion notebook.

Every block below is the code printed in the chapter, in the same order. Run the cells top to bottom; the output should match the book exactly. If it does not, check `requirements.txt` first, then the errata page.

In [ ]:
!pip -q install -r https://raw.githubusercontent.com/USER/from-absolute-zero/main/requirements.txt  # Colab only; skip locally

## Shared setup

Imports and the objects the blocks below reuse. The chapter prints these once and then continues the same session.

In [ ]:
import numpy as np, warnings; warnings.filterwarnings("ignore")
from scipy import stats

## The chapter code

### Block 1  (`c1.py`)

In [ ]:
# Data drift: the distribution of an input feature shifts after
# deployment. The Kolmogorov-Smirnov test, from Chapter 8's hypothesis
# testing toolkit, compares a recent window of live data against the
# distribution the model was trained on.
r = np.random.default_rng(43)
baseline = r.normal(50, 10, 2000)          # the feature's distribution at training time

def ks_test(baseline, live):
    stat, p = stats.ks_2samp(baseline, live)
    return stat, p

print(f"{'week':>6}{'true mean shift':>17}{'KS statistic':>14}{'p-value':>12}{'flagged?':>10}")
for week in range(0, 13):
    shift = week * 0.6                       # the feature drifts gradually, 0.6 units per week
    live = r.normal(50 + shift, 10, 300)      # this week's monitoring window
    stat, p = ks_test(baseline, live)
    flagged = "yes" if p < 0.01 else "no"
    print(f"{week:>6}{shift:>17.1f}{stat:>14.4f}{p:>12.4f}{flagged:>10}")

### Block 2  (`c2.py`)

In [ ]:
# The Population Stability Index bins both distributions the same way
# and compares bin proportions directly, which is what most production
# monitoring dashboards actually report rather than a raw p-value.
def psi(baseline, live, n_bins=10):
    edges = np.quantile(baseline, np.linspace(0, 1, n_bins + 1))
    edges[0], edges[-1] = -np.inf, np.inf
    base_counts, _ = np.histogram(baseline, bins=edges)
    live_counts, _ = np.histogram(live, bins=edges)
    base_pct = np.clip(base_counts / len(baseline), 1e-4, None)
    live_pct = np.clip(live_counts / len(live), 1e-4, None)
    return np.sum((live_pct - base_pct) * np.log(live_pct / base_pct))

print(f"{'week':>6}{'true mean shift':>17}{'PSI':>10}{'interpretation':>18}")
for week in range(0, 13):
    shift = week * 0.6
    live = r.normal(50 + shift, 10, 300)
    score = psi(baseline, live)
    if score < 0.1:
        label = "stable"
    elif score < 0.25:
        label = "moderate shift"
    else:
        label = "major shift"
    print(f"{week:>6}{shift:>17.1f}{score:>10.4f}{label:>18}")

print(f"\nPSI thresholds of 0.1 and 0.25 are the conventional industry cutoffs.")

### Block 3  (`c3.py`)

In [ ]:
# The trap: concept drift changes the relationship between features and
# the outcome, not the features themselves. A monitor watching only the
# input distribution can see nothing wrong at all while the model
# quietly gets every prediction wrong.
r2 = np.random.default_rng(430)

def make_labels(x, true_boundary):
    return (x > true_boundary).astype(int)

x_train = r2.normal(50, 10, 2000)
y_train = make_labels(x_train, true_boundary=50)     # trained decision boundary: x > 50

# a simple threshold "model", fit once at training time and never touched again
model_boundary = 50.0

print(f"{'week':>6}{'feature mean':>14}{'feature KS p-value':>20}{'true boundary':>15}{'model accuracy':>16}")
for week in range(0, 13):
    x_live = r2.normal(50, 10, 500)                   # feature distribution: UNCHANGED, every week
    true_boundary_now = 50 + week * 2.5               # but what defines the outcome keeps moving
    y_live = make_labels(x_live, true_boundary_now)
    pred = (x_live > model_boundary).astype(int)
    acc = (pred == y_live).mean()
    _, p = ks_test(x_train, x_live)
    print(f"{week:>6}{x_live.mean():>14.2f}{p:>20.4f}{true_boundary_now:>15.1f}{acc:>16.4f}")

print(f"\nthe feature distribution never moves, so every distribution-based")
print(f"monitor in Steps 1 and 2 would report perfect stability every week.")

### Block 4  (`c4.py`)

In [ ]:
# The fix for concept drift is to monitor what the model actually gets
# right, not just what it sees. This requires labels to eventually
# arrive, often delayed, but even a small labelled sample each week is
# enough to catch what distribution monitoring cannot see at all.
# Reruns the identical scenario from Step 3, on the identical random
# draws, so the two diagnostics describe the same twelve weeks rather
# than two separately-sampled ones.
r3 = np.random.default_rng(430)
_ = r3.normal(50, 10, 2000)                # advance the stream past x_train, exactly as Step 3 did

print(f"{'week':>6}{'model accuracy':>16}{'flagged (acc < 0.85)?':>23}")
for week in range(0, 13):
    x_live = r3.normal(50, 10, 500)
    true_boundary_now = 50 + week * 2.5
    y_live = make_labels(x_live, true_boundary_now)
    pred = (x_live > model_boundary).astype(int)
    acc = (pred == y_live).mean()
    flagged = "yes -- retrain" if acc < 0.85 else "no"
    print(f"{week:>6}{acc:>16.4f}{flagged:>23}")

print(f"\na performance monitor catches the problem at week 2, when the true")
print(f"boundary has moved only 5 of its eventual 30 units, one sixth of the")
print(f"way there. The feature-distribution test in Step 3 never flags this")
print(f"drift at all across the same twelve weeks.")

### Block 5  (`c5.py`)

In [ ]:
# Serving cost has its own tradeoff: batching several requests together
# raises throughput, since fixed per-batch overhead is shared across
# more predictions, but it raises the latency any single request in
# that batch has to wait for, since it waits for the whole batch to fill.
def simulate_serving(batch_size, per_batch_overhead_ms, per_item_ms, arrival_rate_per_ms):
    # time to fill a batch, on average, given how fast requests arrive
    fill_time = batch_size / arrival_rate_per_ms
    compute_time = per_batch_overhead_ms + batch_size * per_item_ms
    # a request's expected wait: half the fill time (arrives at a random
    # point in the batch-filling window) plus the compute time
    avg_latency = fill_time / 2 + compute_time
    throughput = batch_size / (fill_time + compute_time) * 1000    # predictions per second
    return avg_latency, throughput

print(f"{'batch size':>11}{'avg latency (ms)':>19}{'throughput (pred/s)':>22}")
for batch in (1, 4, 8, 16, 32, 64):
    latency, throughput = simulate_serving(batch, per_batch_overhead_ms=8,
                                           per_item_ms=0.6, arrival_rate_per_ms=0.5)
    print(f"{batch:>11}{latency:>19.2f}{throughput:>22.1f}")

print(f"\nlarger batches serve far more predictions per second, at the cost")
print(f"of a slower response for any single one of them.")